# Unzip and preprocessing

In [1]:
import zipfile
import io
import os
import random
from pathlib import Path
import numpy as np
import cv2
from PIL import Image
from tqdm import tqdm
import pandas as pd  # 💡 新增：為了最後產出 labels.csv

# 正確引入助教給的 hand_preprocess.py 裡面的類別
from hand_preprocess import MediaPipeHandPreprocessor 

# ==================== 🛠️ 10000張滑塊控制面板 ====================
ZIP_FILE_PATH = Path(r"D:/hagrid_project/hagridv2_512.zip")      

# 💡 這次輸出後的資料夾，就是直接給 PyTorch 吃的終極版
OUTPUT_DIR = Path(r"D:/hagrid_project/dataset_v4_processed")     
START_INDEX = 0                                                            

# 5 個專題指定的標準目標手勢
TARGET_CATEGORIES = {'fist', 'like', 'ok', 'one', 'palm'}

# 💡 標籤對照表 (0~5)，給 CSV 紀錄用的
LABEL_MAP = {
    "N_A": 0,
    "fist": 1,
    "like": 2,
    "ok": 3,
    "one": 4,
    "palm": 5,
}

# 🎯 【10,000 張的黃金配額比例】
MAX_PER_TARGET = 25000    # 每個目標類別 1000 張
MAX_PER_NOISE = 863       # 29個雜訊每個 172 張

RANDOM_SEED = 21         # 固定隨機種子碼，確保全局打亂順序一致
# ===============================================================

def main():
    if not ZIP_FILE_PATH.exists():
        print(f"❌ 找不到原始壓縮檔，請檢查路徑：{ZIP_FILE_PATH}")
        return

    preprocessor = MediaPipeHandPreprocessor()
    
    # 💡 建立扁平化的資料夾結構 (不再分 fist/, ok/ 了，全部倒進這兩個)
    crop_dir = OUTPUT_DIR / "crops"
    landmark_dir = OUTPUT_DIR / "landmarks"
    crop_dir.mkdir(parents=True, exist_ok=True)
    landmark_dir.mkdir(parents=True, exist_ok=True)

    print(f"🚀 [一條龍終極流水線] 目錄：{OUTPUT_DIR} | 起始點控制：{START_INDEX}")
    
    saved_counters = {}
    records = [] # 💡 用來記錄每一筆資料，最後轉成 CSV

    with zipfile.ZipFile(ZIP_FILE_PATH, 'r') as z:
        print("🔍 步驟一：正在建立壓縮檔全量索引...")
        all_files = z.namelist()
        
        # 👑 多層路徑智慧探測
        raw_img_files = []
        for f in all_files:
            if f.lower().endswith(('.jpg', '.jpeg', '.png')) and ".ipynb_checkpoints" not in f:
                parts = f.split('/')
                
                matched_cat = None
                for part in reversed(parts):
                    if part and not part.endswith(('.jpg', '.jpeg', '.png')) and part != "hagridv2_512":
                        matched_cat = part
                        break
                
                if matched_cat:
                    raw_img_files.append((f, matched_cat))

        # 動態抓取所有真正的手勢類別來初始化計數器
        all_detected_cats = set(cat_name for _, cat_name in raw_img_files)
        for cat in all_detected_cats:
            saved_counters[cat] = 0

        print(f"📂 成功辨識出 {len(all_detected_cats)} 個手勢類別。原始總圖數：{len(raw_img_files)}")
        
        # 🔥 固定 Seed 全局場景大打亂
        print(f"🎴 正在注入 Seed {RANDOM_SEED} 進行全局打亂...")
        random.seed(RANDOM_SEED)
        random.shuffle(raw_img_files)
        
        # 👑 發動滑塊切片
        sliced_img_files = raw_img_files[START_INDEX:]
        print(f"📐 開炸 MediaPipe 流水線...")

        saved_counter = 0
        for file_path, cat_name in tqdm(sliced_img_files, desc="10000張平衡清洗與格式化中"):
            
            # 🛑 煞車機制
            if cat_name in TARGET_CATEGORIES and saved_counters[cat_name] >= MAX_PER_TARGET:
                continue
            if cat_name not in TARGET_CATEGORIES and saved_counters[cat_name] >= MAX_PER_NOISE:
                continue

            try:
                original_filename = file_path.split('/')[-1]
                base_name = os.path.splitext(original_filename)[0]

                # 💡 洗 Label 邏輯：判定它是 1~5 還是 0 (N_A)
                if cat_name in TARGET_CATEGORIES:
                    label_name = cat_name
                    numeric_label = LABEL_MAP[cat_name]
                else:
                    label_name = "N_A"
                    numeric_label = 0
                
                # 💡 防撞名設計：不管什麼類別，檔名最前面都加上原始資料夾名稱
                final_base_name = f"{cat_name}_{base_name}"

                # 唯讀串流過水 MediaPipe
                img_bytes = z.read(file_path)
                pil_img = Image.open(io.BytesIO(img_bytes))
                
                result = preprocessor.preprocess_image(pil_img)
                if result is None:
                    continue  # 沒抓到手直接放生
                    
                crop_img, landmarks = result
                # crop_resized = cv2.resize(crop_img, (224, 224))

                # 💡 實體寫入到新的扁平化資料夾
                img_out_path = crop_dir / f"{final_base_name}.jpg"
                npy_out_path = landmark_dir / f"{final_base_name}.npy"
                
                cv2.imwrite(str(img_out_path), cv2.cvtColor(crop_img, cv2.COLOR_RGB2BGR))
                np.save(str(npy_out_path), landmarks)

                # 💡 將這筆成功的資料記錄到 CSV 名單中
                records.append({
                    "idx": final_base_name,
                    "original_class_folder": cat_name,
                    "label": numeric_label,
                    "label_name": label_name,
                    "crop_path": str(img_out_path),
                    "landmark_path": str(npy_out_path),
                    "quality": "ok"
                })

                saved_counters[cat_name] += 1
                saved_counter += 1

                # 檢查是否所有類別都集滿了
                all_done = True
                for c in all_detected_cats:
                    limit = MAX_PER_TARGET if c in TARGET_CATEGORIES else MAX_PER_NOISE
                    if saved_counters[c] < limit:
                        all_done = False
                        break
                if all_done:
                    print("\n🎯 賀！本資料集已完美集滿黃金配額！提前收網！")
                    break

            except Exception:
                continue

    # ==================== 📊 最終產出 CSV 與報告 ====================
    print("\n📝 正在生成 PyTorch 專用標籤對照表 (labels.csv)...")
    df = pd.DataFrame(records)
    csv_path = OUTPUT_DIR / "labels.csv"
    df.to_csv(csv_path, index=False)

    total_target_saved = sum(saved_counters[c] for c in TARGET_CATEGORIES if c in saved_counters)
    total_na_saved = sum(saved_counters[c] for c in saved_counters if c not in TARGET_CATEGORIES)
    
    print("\n==================================================")
    print(f"🎉 【一條龍建構完畢】！")
    print(f"📈 產出摘要：")
    print(f"   - 5大目標手勢總計：{total_target_saved} 張照片對")
    print(f"   - 29類均勻 N/A 總計：{total_na_saved} 張照片對 (Class 0)")
    print(f"📁 圖片位置：{crop_dir}")
    print(f"📁 座標位置：{landmark_dir}")
    print(f"🧾 總目錄 CSV：{csv_path}")
    print("==================================================")

if __name__ == "__main__":
    main()

🚀 [一條龍終極流水線] 目錄：D:\hagrid_project\dataset_v4_processed | 起始點控制：0
🔍 步驟一：正在建立壓縮檔全量索引...
📂 成功辨識出 34 個手勢類別。原始總圖數：1086158
🎴 正在注入 Seed 21 進行全局打亂...
📐 開炸 MediaPipe 流水線...


10000張平衡清洗與格式化中:  91%|█████████ | 983274/1086158 [1:57:58<12:20, 138.92it/s]  



🎯 賀！本資料集已完美集滿黃金配額！提前收網！

📝 正在生成 PyTorch 專用標籤對照表 (labels.csv)...

🎉 【一條龍建構完畢】！
📈 產出摘要：
   - 5大目標手勢總計：125000 張照片對
   - 29類均勻 N/A 總計：25027 張照片對 (Class 0)
📁 圖片位置：D:\hagrid_project\dataset_v4_processed\crops
📁 座標位置：D:\hagrid_project\dataset_v4_processed\landmarks
🧾 總目錄 CSV：D:\hagrid_project\dataset_v4_processed\labels.csv


# False Crop detect

In [2]:
import numpy as np
import pandas as pd
import shutil
from pathlib import Path

# 1. 設定路徑
data_dir = Path("D:/hagrid_project/dataset_v4_processed")
csv_path = data_dir / "labels.csv"
need_check_dir = data_dir / "need_check"

# 建立 need_check 資料夾
if need_check_dir.exists():
    shutil.rmtree(need_check_dir) # 如果之前有跑過，先清空
need_check_dir.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(csv_path)

# --- 關鍵修正：路徑自動修正 ---
def fix_path(path_str):
    filename = Path(path_str).name
    if "landmark" in str(path_str):
        return str(data_dir / "landmarks" / filename)
    else:
        return str(data_dir / "crops" / filename)

df['landmark_path'] = df['landmark_path'].apply(fix_path)
df['crop_path'] = df['crop_path'].apply(fix_path)

def dist(a, b):
    return np.linalg.norm(a - b)

def get_finger_states(lm):
    wrist = lm[0]
    palm_scale = dist(lm[0], lm[9]) + 1e-6
    # 寬鬆版：threshold 設為 0.0
    def is_extended(mcp, pip, dip, tip, threshold=0.0):
        return dist(lm[tip], wrist) > dist(lm[pip], wrist) + threshold * palm_scale

    return {
        "thumb": dist(lm[4], (lm[0]+lm[5]+lm[9]+lm[13]+lm[17])/5) > 0.45 * palm_scale,
        "index": is_extended(5, 6, 7, 8),
        "middle": is_extended(9, 10, 11, 12),
        "ring": is_extended(13, 14, 15, 16),
        "pinky": is_extended(17, 18, 19, 20),
    }

def check_expected(label_name, lm):
    states = get_finger_states(lm)
    thumb, index = states["thumb"], states["index"]
    middle, ring, pinky = states["middle"], states["ring"], states["pinky"]
    extended_count = sum(states.values())
    palm_scale = dist(lm[0], lm[9]) + 1e-6
    thumb_index_tip_dist = dist(lm[4], lm[8]) / palm_scale

    # if label_name == "fist":
    #     # fist: 大部分手指應該是彎的
    #     ok = extended_count <= 1

    # elif label_name == "like":
    #     # like: 拇指伸出，其他四指大多彎曲
    #     ok = thumb and (not index) and (not middle) and (not ring) and (not pinky)

    # elif label_name == "one":
    #     # one: 食指伸出，其他大多彎曲
    #     ok = index and (not middle) and (not ring) and (not pinky)

    # elif label_name == "palm":
    #     # palm: 至少四根手指伸出
    #     ok = extended_count >= 4

    # elif label_name == "ok":
    #     # ok: 拇指尖和食指尖靠近
    #     # middle/ring/pinky 通常會伸出，但先不要設太嚴
    #     ok = thumb_index_tip_dist < 0.45

    # else:
    #     # N/A 不檢查 target pattern
    #     ok = True


    if label_name == "fist":
        ok = extended_count <= 2 # 允許微鬆的拳頭
    elif label_name == "like":
        ok = thumb and (not index) and (not middle) # 只看拇指、食指、中指
    elif label_name == "one":
        ok = index and (not middle) and (not ring)
    elif label_name == "palm":
        ok = extended_count >= 3 # 至少3根手指
    elif label_name == "ok":
        ok = thumb_index_tip_dist < 0.8 # 巨幅放寬 OK 的距離
    else:
        ok = True

    return ok

suspicious_ids = []

print("開始掃描可疑圖片...")
for row in df.itertuples():
    if row.label_name == "N_A":
        continue
        
    lm = np.load(row.landmark_path)
    if not check_expected(row.label_name, lm):
        suspicious_ids.append(row.idx)
        
        # 依照標籤建立子資料夾，方便人類看
        label_dir = need_check_dir / row.label_name
        label_dir.mkdir(exist_ok=True)
        
        # 將可疑圖片複製過去，檔名加上 idx_ 原檔名
        dest_path = label_dir / f"{row.idx}_{Path(row.crop_path).name}"
        shutil.copy(row.crop_path, dest_path)

print(f"掃描完成！共有 {len(df)} 張目標手勢。")
print(f"抓出 {len(suspicious_ids)} 張可疑圖片，已複製到 {need_check_dir} 中。")

開始掃描可疑圖片...
掃描完成！共有 150027 張目標手勢。
抓出 8067 張可疑圖片，已複製到 D:\hagrid_project\dataset_v4_processed\need_check 中。


In [3]:
import pandas as pd
from pathlib import Path

# 1. 設定你的 CSV 路徑
csv_path = Path("D:/hagrid_project/dataset_v4_processed/labels.csv")
output_path = Path("D:/hagrid_project/dataset_v4_processed/labels_fixed.csv")

# 2. 讀取資料表
df = pd.read_csv(csv_path)

# 3. 執行「一鍵轉 N/A」
# 假設 suspicious_ids 是你剛才透過規則抓出來的 324 個嫌疑犯索引清單
# 如果你剛好沒有那個清單變數，也可以直接用過濾器重新篩選一次
# 這裡我們將所有 label 改為 0，label_name 改為 N_A
df.loc[df['idx'].isin(suspicious_ids), 'label'] = 0
df.loc[df['idx'].isin(suspicious_ids), 'label_name'] = 'N_A'

# 4. 存檔
df.to_csv(output_path, index=False)

print(f"✅ 大功告成！")
print(f"已將 {len(suspicious_ids)} 筆疑似垃圾資料轉為 N_A (Class 0)")
print(f"新檔案已儲存至: {output_path}")

✅ 大功告成！
已將 8067 筆疑似垃圾資料轉為 N_A (Class 0)
新檔案已儲存至: D:\hagrid_project\dataset_v4_processed\labels_fixed.csv


In [4]:
import shutil

# 想要壓縮的目標資料夾路徑 (可以是相對路徑或絕對路徑)
folder_to_compress = 'D:/hagrid_project/dataset_v4_processed'

# 輸出的壓縮檔名稱與路徑 (不需包含 .zip 副檔名，系統會自動加上)
output_filename = 'D:/hagrid_project/dataset_v4_processed'

# 執行壓縮
# 參數1: 輸出檔名
# 參數2: 壓縮格式 (支援 'zip', 'tar', 'gztar', 'bztar', 'xztar')
# 參數3: 要被壓縮的資料夾路徑
shutil.make_archive(output_filename, 'zip', folder_to_compress)

print("壓縮完成！")

壓縮完成！
